# Transpilación de SVM, Scaler y Parámetros Dinámicos a C++ para Microcontrolador (ESP32-S3)

Este notebook se encarga de:
1. Transpilar el modelo SVM a C++ con parches de compatibilidad para microcontroladores Xtensa LX7 (`va_start(w, x)` y `#include <cstdint>`, `#include <cmath>`).
2. Exportar la configuración dinámica del modelo (`model_config.h`) con la dimensión de ventana ($W, S$), la calibración de sesión ($	ext{MVC}$, $\sigma_{\text{noise}}$) y los parámetros del escalador $Z$-Score directamente a `firmware/Classifier/include/`.

In [7]:
import os
import joblib
import json
import re
from pathlib import Path
from micromlgen import port

# Función para buscar y cargar el archivo .env
def load_env_variables():
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()

✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


## 1. Transpilar la SVM a C++ para ESP32-S3

In [8]:
# 1. Transpilar la SVM a C++ para ESP32-S3
models_dir = os.environ["MODELS_ML_PROTO"]
svm_path = os.path.join(models_dir, "svm_model.bin")
classifier_include_dir = os.path.join(os.environ["PROJECT_ROOT"], "firmware", "Classifier", "include")

print(f"📖 Cargando modelo SVM desde: {svm_path} ...")
svm_model = joblib.load(svm_path)

print("⚡ Transpilando modelo SVM a C++ usando micromlgen...")
cpp_code = port(svm_model)

# Corrección de compatibilidad en micromlgen para microcontroladores Xtensa / GCC
# 1. Inyectar <cstdint> y <cmath>
# 2. Corregir va_start(w, N) -> va_start(w, x) para evitar desalineación de pila en Xtensa
headers = ["#include <cstdint>", "#include <cmath>", ""]
cpp_code = "\n".join(headers) + "\n" + cpp_code
cpp_code = re.sub(r"va_start\s*\(\s*w\s*,\s*\d+\s*\);", "va_start(w, x);", cpp_code)

svm_header_path = os.path.join(classifier_include_dir, "svm_model.h")
os.makedirs(classifier_include_dir, exist_ok=True)

with open(svm_header_path, "w", encoding="utf-8") as f:
    f.write("/*\n")
    f.write(" * =============================================================\n")
    f.write(" *  svm_model.h — Modelo SVM autogenerado por micromlgen\n")
    f.write(" * =============================================================\n")
    f.write(" */\n\n")
    f.write("#pragma once\n")
    f.write(cpp_code)

print(f"✅ Cabecera C++ guardada exitosamente en: {svm_header_path}")

📖 Cargando modelo SVM desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/ml/svm_model.bin ...
⚡ Transpilando modelo SVM a C++ usando micromlgen...
✅ Cabecera C++ guardada exitosamente en: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/svm_model.h


## 2. Exportar la Configuración Dinámica y Parámetros del Scaler (`model_config.h`)

In [9]:
# 2. Cargar StandardScaler
scaler_path = os.path.join(models_dir, "feature_scaler.bin")
if not os.path.exists(scaler_path):
    alt_path = os.path.join(os.environ.get("PROCESSED_VECTOR_CLASSIC_PROTO", ""), "features_scaler.bin")
    if os.path.exists(alt_path):
        scaler_path = alt_path

print(f"📖 Cargando StandardScaler desde: {scaler_path} ...")
feature_scaler = joblib.load(scaler_path)

means = feature_scaler.mean_
stds = feature_scaler.scale_

means_str = ", ".join(f"{m:.8f}f" for m in means)
stds_str = ", ".join(f"{s:.8f}f" for s in stds)

# 3. Leer metadatos de sesión (MVC y Ruido Basal)
raw_proto_dir = Path(os.environ["RAW_DATA_PROTO"])
meta_files = sorted(list(raw_proto_dir.glob("S11/*_metadata.json")))
if not meta_files:
    meta_files = sorted(list(raw_proto_dir.glob("*/*_metadata.json")))

if meta_files:
    with open(meta_files[0], "r", encoding="utf-8") as f:
        meta = json.load(f)
    mvc_voltage = meta.get("calibration", {}).get("mvc_voltage_v", 189.84517)
    noise_std = meta.get("calibration", {}).get("normalized_noise_std", 0.07440)
else:
    mvc_voltage = 189.84517
    noise_std = 0.07440

# Dimensiones de ventana
W = 200
S = 100

model_config_path = os.path.join(classifier_include_dir, "model_config.h")
scaler_header_path = os.path.join(classifier_include_dir, "scaler_params.h")

model_config_content = f"""#pragma once
/*
 * =============================================================
 *  model_config.h — Configuración Dinámica Autogenerada
 *  Generado automáticamente desde el pipeline de Python
 * =============================================================
 */

// Parámetros de Ventana
#define MODEL_WINDOW_SIZE      {W}
#define MODEL_WINDOW_STRIDE    {S}

// Calibración de Fábrica de la Sesión
#define MODEL_MVC_VOLTAGE_V    {mvc_voltage:.5f}f
#define MODEL_NOISE_THRESHOLD  {noise_std:.5f}f

// Parámetros del StandardScaler (MAV, RMS, WL, ZC, SSC, VAR)
const float feature_means[6] = {{ {means_str} }};
const float feature_stds[6]  = {{ {stds_str} }};
"""

with open(model_config_path, "w", encoding="utf-8") as f:
    f.write(model_config_content)

# Mantener scaler_params.h por retrocompatibilidad
with open(scaler_header_path, "w", encoding="utf-8") as f:
    f.write(model_config_content)

print(f"✅ Configuración dinámica guardada exitosamente en: {model_config_path}")
print(f"   * Window Size (W):    {W} muestras")
print(f"   * Window Stride (S):  {S} muestras")
print(f"   * MVC Voltage:        {mvc_voltage:.5f} V")
print(f"   * Noise Threshold:    {noise_std:.5f}")

📖 Cargando StandardScaler desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/ml/feature_scaler.bin ...
✅ Configuración dinámica guardada exitosamente en: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/model_config.h
   * Window Size (W):    200 muestras
   * Window Stride (S):  100 muestras
   * MVC Voltage:        189.84517 V
   * Noise Threshold:    0.07440
